# 01 — Classes, `__init__`, and `self`

A **class** is a blueprint. An **instance** is an actual object built from that blueprint.

We start by writing the smallest possible class, then build the mental model for what `self` is and how attribute lookup works.

## The minimum class

In [1]:
class Dog:
    pass

# Create two instances. They are distinct objects with the same blueprint.
fido = Dog()
rex  = Dog()

print(type(fido))
print(fido is rex)         # False — different objects
print(isinstance(fido, Dog))

<class '__main__.Dog'>
False
True


## `__init__` — set up the instance

When you write `Dog("Fido", 3)`, Python:

1. Creates a new empty `Dog` instance.
2. Calls `__init__(self, "Fido", 3)` — `self` is the new instance.
3. Returns the instance.

`__init__` is a *dunder method* (double-underscore) — Python's name for methods with special meaning. `self` is just a parameter name (convention; you *could* call it `this`, but don't).

In [2]:
class Dog:
    def __init__(self, name, age):
        # These lines attach attributes TO THE INSTANCE.
        self.name = name
        self.age = age

fido = Dog("Fido", 3)
rex  = Dog("Rex", 5)

print(fido.name, fido.age)
print(rex.name, rex.age)

Fido 3
Rex 5


### What `self` actually is

`self` is the *instance the method was called on*. Python passes it automatically: `fido.bark()` is sugar for `Dog.bark(fido)`.

In [3]:
class Dog:
    def __init__(self, name):
        self.name = name

    def bark(self):
        # When `fido.bark()` runs, self IS fido.
        print(f"{self.name} says woof!")

fido = Dog("Fido")
fido.bark()

# Equivalent, ugly, never write this in real code — but useful to see:
Dog.bark(fido)

Fido says woof!
Fido says woof!


## Instance attributes vs class attributes — the big one

Anything you assign with `self.x = ...` is an **instance attribute**: it lives on one specific instance.

Anything you assign **inside the class body** (not inside a method) is a **class attribute**: there's *one copy*, shared by every instance.

In [4]:
class Dog:
    species = "Canis familiaris"     # class attribute — shared
    legs = 4                          # class attribute

    def __init__(self, name):
        self.name = name              # instance attribute — per-dog

fido = Dog("Fido")
rex  = Dog("Rex")

print(fido.name, fido.species)
print(rex.name,  rex.species)

# All dogs see the same `species` because lookup falls back to the class:
print(fido.species is rex.species)
print(Dog.species)

Fido Canis familiaris
Rex Canis familiaris
True
Canis familiaris


### Attribute lookup order

When you write `fido.species`, Python looks:
1. On the **instance** (`fido.__dict__`).
2. On the **class** (`Dog.__dict__`).
3. On any **parent classes** (we'll get to inheritance in folder 04).

First match wins. This is why a class attribute can be *shadowed* by an instance attribute.

In [5]:
fido.species = "Good Boy"      # creates an INSTANCE attribute on fido — does NOT touch Dog.species
print("fido :", fido.species)
print("rex  :", rex.species)
print("class:", Dog.species)

print("fido.__dict__:", fido.__dict__)   # shows fido's own attributes
print("rex .__dict__:", rex.__dict__)

fido : Good Boy
rex  : Canis familiaris
class: Canis familiaris
fido.__dict__: {'name': 'Fido', 'species': 'Good Boy'}
rex .__dict__: {'name': 'Rex'}


## The class-attribute *mutable* trap

Using a mutable class attribute (like `[]` or `{}`) is the OOP twin of the mutable-default trap from `02_functions_deep`.

In [6]:
class Player:
    inventory = []          # BAD — class attribute, shared by ALL players

a = Player()
b = Player()
a.inventory.append("sword")
print("a:", a.inventory)
print("b:", b.inventory)    # also has 'sword'
print("same object?", a.inventory is b.inventory)

a: ['sword']
b: ['sword']
same object? True


In [7]:
class Player:
    def __init__(self):
        self.inventory = []   # GOOD — a fresh list PER INSTANCE

a = Player()
b = Player()
a.inventory.append("sword")
print("a:", a.inventory)
print("b:", b.inventory)

a: ['sword']
b: []


**Rule:** if the attribute is mutable, assign it in `__init__`. Class attributes are fine for *immutable* constants (numbers, strings, tuples, frozensets) — like our `species` example.

## Default values in `__init__` — same rule as functions

Constructors are just functions. The mutable-default rule applies.

In [8]:
class User:
    def __init__(self, name, tags=None):
        self.name = name
        self.tags = tags if tags is not None else []     # NOT tags=[]

u1 = User("alice")
u2 = User("bob")
u1.tags.append("admin")
print(u1.tags, u2.tags)     # ['admin']  []

['admin'] []


## Mini-exercise

1. Write a `Book` class with `title`, `author`, and `pages`. Add a `summary()` method that returns `f"{title} by {author} ({pages}p)"`.
2. Add a `total_books = 0` *class attribute* and increment it in `__init__`. Create three books and print `Book.total_books`. Is this OK to do with an `int`? Why? (Hint: int is immutable, but reassignment via `Book.total_books += 1` is a class-level rebind, not a per-instance shadow.)
3. Predict, then run:
   ```python
   class C:
       tags = []
       def __init__(self, name):
           self.name = name
       def tag(self, t):
           self.tags.append(t)     # APPENDS to the class attribute!
   x = C("x"); y = C("y")
   x.tag("a"); y.tag("b")
   print(x.tags, y.tags, C.tags)
   ```

In [1]:
# your solutions here
class C:
       tags = []
       def __init__(self, name):
           self.name = name
       def tag(self, t):
           self.tags.append(t)     # APPENDS to the class attribute!
x = C("x"); y = C("y")
x.tag("a"); y.tag("b")
print(x.tags, y.tags, C.tags)   


['a', 'b'] ['a', 'b'] ['a', 'b']


**Takeaways**

- `class` = blueprint, `Class(...)` = instance creation, `__init__` = setup.
- `self` is the instance; `fido.bark()` is sugar for `Dog.bark(fido)`.
- Instance attributes (`self.x = ...`) are per-object. Class attributes are shared.
- **Never** use a mutable class attribute when you wanted per-instance state.

Next: [02_methods.ipynb](02_methods.ipynb) — the three method kinds and pretty printing.